In [15]:
from __future__ import annotations
from collections.abc import Sequence
from functools import total_ordering
from random import sample, seed
from itertools import permutations
from math import factorial

In [16]:
NUM_CHILDREN = 5
NUM_CARACTERISTICS = 20
seed(42)  # same random children at every run

In [17]:
@total_ordering
class Child:
    _name: str
    _caracteristics: tuple[bool, ...]

    def __init__(self, name: str, caracteristics: Sequence[bool]):
        self._name = name
        self._caracteristics = tuple(caracteristics)

    # Hamming distance: number of characteristics that are different
    def difference(self, other: Child) -> int:
        return sum(s != o for s, o in zip(self._caracteristics, other._caracteristics))

    def __str__(self) -> str:
        return f"{self._name}:{''.join('+' if c else '-' for c in self._caracteristics)}"

    def __repr__(self) -> str:
        return f"{self._name}"

    # comparison based on id(), needed by normalize
    def __eq__(self, other):
        return id(self) == id(other)

    def __lt__(self, other):
        return id(self) < id(other)

In [18]:
# create the children with random characteristics
CHILDREN = list()

for c in range(NUM_CHILDREN):
    name = f"Child{c}"
    CHILDREN.append(
        Child(
            name,
            sample(
                [True, False], k=NUM_CARACTERISTICS, counts=[NUM_CARACTERISTICS, NUM_CARACTERISTICS]
            ),
        )
    )

In [19]:
for c in CHILDREN:
    print(c)

Child0:++++++++--++--++----
Child1:--+-+++--+++--+++-+-
Child2:-++--+-++--+-+-+--+-
Child3:++++-+--++---+---++-
Child4:+-+---++----++--++++


In [20]:
# difference between every pair of children
for a in CHILDREN:
    for b in CHILDREN:
        if a._name < b._name:
            print(a._name, "-", b._name, ":", a.difference(b))

Child0 - Child1 : 7
Child0 - Child2 : 9
Child0 - Child3 : 12
Child0 - Child4 : 14
Child1 - Child2 : 10
Child1 - Child3 : 13
Child1 - Child4 : 13
Child2 - Child3 : 7
Child2 - Child4 : 11
Child3 - Child4 : 10


In [21]:
# sum of the differences between neighbours (each pair counted twice)
def fitness(solution: Sequence[Child]) -> int:
    sol = [CHILDREN[0]] + list(solution)  # Child0 is always the first
    fitness = 0
    for cp, c, cn in zip(sol, sol[1:] + sol[0:1], sol[2:] + sol[0:3]):
        fitness += c.difference(cp)
        fitness += c.difference(cn)
    return fitness


# a circle and its reverse are the same: always keep the same one
def normalize(solution: Sequence[Child]) -> list[Child]:
    if solution[0] > solution[-1]:
        return list(solution[::-1])
    else:
        return list(solution)


# swap two random children
def tweak(solution: Sequence[Child]) -> list[Child]:
    i1, i2 = sample(range(len(solution)), k=2)
    new_solution = list(solution)
    new_solution[i1] = solution[i2]
    new_solution[i2] = solution[i1]
    return normalize(new_solution)

In [22]:
# example: fitness of the circle Child0-Child1-Child2-Child3-Child4
solution = CHILDREN[1:]
circle = [CHILDREN[0]] + list(solution)

total = 0
for i in range(len(circle)):
    a = circle[i]
    b = circle[(i + 1) % len(circle)]  # the last child is next to the first one
    print(a._name, "-", b._name, ":", a.difference(b))
    total += a.difference(b)

print("sum of the differences:", total)
print("fitness:", fitness(solution), "(= 2 x sum, each pair is counted twice)")

Child0 - Child1 : 7
Child1 - Child2 : 10
Child2 - Child3 : 7
Child3 - Child4 : 10
Child4 - Child0 : 14
sum of the differences: 48
fitness: 96 (= 2 x sum, each pair is counted twice)


In [23]:
# example: tweak
print("before:", solution, fitness(solution))
new_solution = tweak(solution)
print("after: ", new_solution, fitness(new_solution))

before: [Child1, Child2, Child3, Child4] 96
after:  [Child3, Child2, Child1, Child4] 112


In [24]:
# example: a solution and its reverse give the same list
s = list(CHILDREN[1:])
print(s, "->", normalize(s))
print(s[::-1], "->", normalize(s[::-1]))
print("same fitness:", fitness(s), fitness(s[::-1]))

[Child1, Child2, Child3, Child4] -> [Child1, Child2, Child3, Child4]
[Child4, Child3, Child2, Child1] -> [Child1, Child2, Child3, Child4]
same fitness: 96 96


In [25]:
# exhaustive search: try every permutation and keep the best
best_solution = None
best_fitness = 0

for solution in permutations(CHILDREN[1:]):
    print(solution, fitness(solution))
    if fitness(solution) > best_fitness:
        best_solution = solution
        best_fitness = fitness(solution)

print()
print("BEST:", best_solution, best_fitness)

(Child1, Child2, Child3, Child4) 96
(Child1, Child2, Child4, Child3) 100
(Child1, Child3, Child2, Child4) 104
(Child1, Child3, Child4, Child2) 100
(Child1, Child4, Child2, Child3) 100
(Child1, Child4, Child3, Child2) 92
(Child2, Child1, Child3, Child4) 112
(Child2, Child1, Child4, Child3) 108
(Child2, Child3, Child1, Child4) 112
(Child2, Child3, Child4, Child1) 92
(Child2, Child4, Child1, Child3) 116
(Child2, Child4, Child3, Child1) 100
(Child3, Child1, Child2, Child4) 120
(Child3, Child1, Child4, Child2) 116
(Child3, Child2, Child1, Child4) 112
(Child3, Child2, Child4, Child1) 100
(Child3, Child4, Child1, Child2) 108
(Child3, Child4, Child2, Child1) 100
(Child4, Child1, Child2, Child3) 112
(Child4, Child1, Child3, Child2) 112
(Child4, Child2, Child1, Child3) 120
(Child4, Child2, Child3, Child1) 104
(Child4, Child3, Child1, Child2) 112
(Child4, Child3, Child2, Child1) 96

BEST: (Child3, Child1, Child2, Child4) 120


In [26]:
# hill climbing: start from a random solution
solution = normalize(sample(CHILDREN[1:], k=NUM_CHILDREN - 1))
print("start:", solution, fitness(solution))

for step in range(100):
    new_solution = tweak(solution)
    if fitness(new_solution) > fitness(solution):
        print("step", step, ":", new_solution, fitness(new_solution), "better!")
    # keep the new solution if it is not worse
    if fitness(new_solution) >= fitness(solution):
        solution = new_solution

print("final:", solution, fitness(solution))
print("exhaustive search:", best_fitness)

start: [Child1, Child3, Child2, Child4] 104
step 0 : [Child4, Child1, Child3, Child2] 112 better!
step 4 : [Child3, Child1, Child2, Child4] 120 better!
final: [Child3, Child1, Child2, Child4] 120
exhaustive search: 120


In [27]:
# number of solutions with n children: (n-1)!
for n in [5, 8, 10, 15, 20, 30]:
    print(n, "children:", factorial(n - 1), "solutions")

5 children: 24 solutions
8 children: 5040 solutions
10 children: 362880 solutions
15 children: 87178291200 solutions
20 children: 121645100408832000 solutions
30 children: 8841761993739701954543616000000 solutions


In [28]:
# 30 children: too many solutions to try them all, so we use only hill climbing
NUM_CHILDREN = 30

CHILDREN = list()
for c in range(NUM_CHILDREN):
    name = f"Child{c}"
    CHILDREN.append(
        Child(
            name,
            sample(
                [True, False], k=NUM_CARACTERISTICS, counts=[NUM_CARACTERISTICS, NUM_CARACTERISTICS]
            ),
        )
    )

random_solution = normalize(sample(CHILDREN[1:], k=NUM_CHILDREN - 1))
print("a random solution:", fitness(random_solution))

# hill climbing with 5 random restarts
results = []
for run in range(5):
    solution = normalize(sample(CHILDREN[1:], k=NUM_CHILDREN - 1))
    for step in range(5_000):
        new_solution = tweak(solution)
        if fitness(new_solution) >= fitness(solution):
            solution = new_solution
    print("run", run + 1, ":", fitness(solution))
    results.append(fitness(solution))

print("best:", max(results))
print("upper bound (every pair differs on every characteristic):",
      2 * NUM_CHILDREN * NUM_CARACTERISTICS)

a random solution: 624
run 1 : 804
run 2 : 800
run 3 : 816
run 4 : 796
run 5 : 812
best: 816
upper bound (every pair differs on every characteristic): 1200
